In [7]:
!pip install -q -U pyspark "qdrant-client>=1.12.0" "httpx==0.27.2" \
    "langchain<1.0" "langchain-core<1.0" "langchain-community<1.0" \
    "langchain-groq<1.0" "langchain-qdrant" "langchain-huggingface" \
    "langchain-text-splitters<1.0" sentence-transformers gradio

import os
import random
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)

print("[✓] Packages installed.")


spark = SparkSession.builder \
    .appName("OmniCart-Analytics-Session") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print(f"[✓] Spark session active. Version: {spark.version}")


cities = ["Lahore", "Karachi", "Islamabad", "Faisalabad", "Rawalpindi"]
categories = ["Electronics", "Apparel", "Home Decor", "Groceries", "Fitness"]
product_pool = {
    "Electronics": ["Smartphone X", "Wireless Buds", "Smart 4K TV", "Mechanical Keyboard"],
    "Apparel": ["Slim Fit Denim", "Classic Hoodie", "Puffer Jacket", "Air Running Shoes"],
    "Home Decor": ["Ergonomic Desk Lamp", "Memory Foam Pillow", "Scented Candle Set", "Ceramic Vase"],
    "Groceries": ["Organic Coffee Beans", "Premium Basmati Rice", "Extra Virgin Olive Oil"],
    "Fitness": ["Adjustable Dumbbell", "Yoga Mat", "Resistance Bands", "Smart Fitness Tracker"]
}

start_date = datetime(2026, 1, 1)
transactions_data = []

for i in range(1, 5001):
    city = random.choice(cities)
    category = random.choice(categories)
    product = random.choice(product_pool[category])
    quantity = random.randint(1, 4)
    price_per_unit = round(random.uniform(15.0, 1200.0), 2)
    total_amount = round(quantity * price_per_unit, 2)
    timestamp = start_date + timedelta(
        days=random.randint(0, 180), hours=random.randint(0, 23), minutes=random.randint(0, 59)
    )
    transactions_data.append((
        f"TXN_{100000+i}", f"USR_{random.randint(1000, 2500)}", product, category,
        quantity, price_per_unit, total_amount, city, timestamp
    ))

txn_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("price_per_unit", DoubleType(), False),
    StructField("total_amount", DoubleType(), False),
    StructField("location_city", StringType(), False),
    StructField("timestamp", TimestampType(), False)
])

df_transactions = spark.createDataFrame(transactions_data, schema=txn_schema)
df_transactions.createOrReplaceTempView("sales_records")

print(f"[✓] Registered 'sales_records' view with {df_transactions.count()} rows.")


spark.sql("""
    SELECT location_city, category, SUM(total_amount) AS revenue, COUNT(*) AS n_sales
    FROM sales_records GROUP BY location_city, category
    ORDER BY revenue DESC LIMIT 5
""").show()

[✓] Packages installed.
[✓] Spark session active. Version: 4.2.0
[✓] Registered 'sales_records' view with 5000 rows.
+-------------+-----------+------------------+-------+
|location_city|   category|           revenue|n_sales|
+-------------+-----------+------------------+-------+
|       Lahore|    Fitness|385109.26999999996|    245|
|   Faisalabad| Home Decor|         358522.08|    226|
|      Karachi|Electronics|353366.96999999986|    212|
|    Islamabad|Electronics|347530.29999999993|    206|
|       Lahore|Electronics|346647.01999999996|    222|
+-------------+-----------+------------------+-------+



In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

vendor_data = [
    ("VEN_001", "Lahore Textiles Co",      "Lahore",     420,  9,  12.4),
    ("VEN_002", "Karachi Electronics Hub", "Karachi",    610, 14,  8.1),
    ("VEN_003", "Fast Fashion Lahore",     "Lahore",     380, 11, 61.5),
    ("VEN_004", "Islamabad Home Goods",    "Islamabad",  295,  3, 10.2),
    ("VEN_005", "Rawalpindi Fitness Gear", "Rawalpindi", 340,  4, 15.8),
    ("VEN_006", "Faisalabad Grocers",      "Faisalabad", 275,  5, 22.0),
    ("VEN_007", "Karachi Apparel Direct",  "Karachi",    455, 15, 55.9),
    ("VEN_008", "Lahore Smart Devices",    "Lahore",     500,  6,  9.7),
    ("VEN_009", "Islamabad Wellness",      "Islamabad",  210,  2, 11.1),
    ("VEN_010", "Faisalabad Denim Works",  "Faisalabad", 330,  4, 18.3),
]

vendor_schema = StructType([
    StructField("vendor_id", StringType(), False),
    StructField("vendor_name", StringType(), False),
    StructField("city", StringType(), False),
    StructField("total_orders", IntegerType(), False),
    StructField("defect_orders", IntegerType(), False),
    StructField("avg_tracking_upload_hours", DoubleType(), False),
])

df_vendors = spark.createDataFrame(vendor_data, schema=vendor_schema)

from pyspark.sql.functions import round as spark_round
df_vendors = df_vendors.withColumn(
    "defect_rate_pct",
    spark_round((df_vendors.defect_orders / df_vendors.total_orders) * 100, 2)
)

df_vendors.createOrReplaceTempView("vendor_performance")

print(f"Registered 'vendor_performance' view with {df_vendors.count()} vendors.")
spark.sql("""
    SELECT vendor_id, vendor_name, city, defect_rate_pct, avg_tracking_upload_hours
    FROM vendor_performance
    ORDER BY defect_rate_pct DESC
""").show(truncate=False)

Registered 'vendor_performance' view with 10 vendors.
+---------+-----------------------+----------+---------------+-------------------------+
|vendor_id|vendor_name            |city      |defect_rate_pct|avg_tracking_upload_hours|
+---------+-----------------------+----------+---------------+-------------------------+
|VEN_007  |Karachi Apparel Direct |Karachi   |3.3            |55.9                     |
|VEN_003  |Fast Fashion Lahore    |Lahore    |2.89           |61.5                     |
|VEN_002  |Karachi Electronics Hub|Karachi   |2.3            |8.1                      |
|VEN_001  |Lahore Textiles Co     |Lahore    |2.14           |12.4                     |
|VEN_006  |Faisalabad Grocers     |Faisalabad|1.82           |22.0                     |
|VEN_010  |Faisalabad Denim Works |Faisalabad|1.21           |18.3                     |
|VEN_008  |Lahore Smart Devices   |Lahore    |1.2            |9.7                      |
|VEN_005  |Rawalpindi Fitness Gear|Rawalpindi|1.18      

In [9]:
import os
import pandas as pd
from pyspark.sql.functions import col, pandas_udf, explode
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore, RetrievalMode
from langchain_core.documents import Document


print("Loading local embedding model...")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"}
)
print("Embedding model ready (768-dim vectors).")


doc_data = [
    ("shipping_policy.txt", "OmniCart Shipping Guidelines: Standard delivery takes 3 to 5 business days for major metropolitan hubs. Expedited shipping options guarantee overnight shipping for orders placed before 2:00 PM PKT. Shipping fees are dynamically calculated based on dimensional weight and location zone. Orders exceeding 5,000 PKR qualify for completely free standard shipping across Pakistan."),
    ("faq_electronics.txt", "Smart-Device FAQ: If your wireless buds fail to pair with your smartphone, place both earbuds back inside the charging case and hold the rear synchronization button down for 10 full seconds until the LED pulses amber. This triggers a hardware factory reset. Do not submerge the case in water as it lacks an IPX7 water-resistance rating."),
    ("vendor_agreement.txt", "OmniCart Merchant Terms: All third-party marketplace vendors must maintain an order defect rate (ODR) strictly below 1.5% to avoid immediate storefront suspension. Payout cycles occur bi-weekly on the 1st and 15th of every month via direct bank transfer. Failure to upload valid tracking information within 48 hours of order confirmation will penalize your search visibility rating.")
]

df_raw_docs = spark.createDataFrame(doc_data, schema=["doc_source", "raw_content"])


chunk_output_schema = ArrayType(
    StructType([
        StructField("chunk_id", StringType(), False),
        StructField("text", StringType(), False)
    ])
)

@pandas_udf(chunk_output_schema)
def split_text_parallel_udf(contents: pd.Series) -> pd.Series:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=30)
    results = []
    for text_block in contents:
        chunks = splitter.split_text(text_block)
        results.append([{"chunk_id": f"c_{i}", "text": c} for i, c in enumerate(chunks)])
    return pd.Series(results)

print("Running distributed chunking...")
df_processed = df_raw_docs.withColumn("chunks", split_text_parallel_udf(col("raw_content")))

df_final_chunks = df_processed.withColumn("single_chunk", explode(col("chunks"))).select(
    col("doc_source"),
    col("single_chunk.chunk_id").alias("chunk_id"),
    col("single_chunk.text").alias("text")
)

local_chunks = df_final_chunks.collect()
print(f"Produced {len(local_chunks)} chunks across {len(doc_data)} documents.")

langchain_docs = [
    Document(
        page_content=row["text"],
        metadata={"source": row["doc_source"], "chunk_id": row["chunk_id"]}
    )
    for row in local_chunks
]

vector_store = QdrantVectorStore.from_documents(
    langchain_docs,
    embedding=embeddings,
    location=":memory:",
    collection_name="omnicart_knowledge_base",
    retrieval_mode=RetrievalMode.DENSE,
)
print("Qdrant in-memory vector store built.")

query = "What happens if a seller takes 3 days to upload a tracking number?"
matched_docs = vector_store.similarity_search(query, k=1)
for d in matched_docs:
    print(f"\n[MATCH] Source: {d.metadata['source']}")
    print(f"Content: \"{d.page_content}\"")

Loading local embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model ready (768-dim vectors).
Running distributed chunking...
Produced 8 chunks across 3 documents.
Qdrant in-memory vector store built.

[MATCH] Source: vendor_agreement.txt
Content: "within 48 hours of order confirmation will penalize your search visibility rating."


In [10]:
from google.colab import userdata
groq_api_key = userdata.get('Groq_API_Keys')

In [11]:
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain.tools import Tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

groq_api_key = userdata.get('Groq_API_Keys')

llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.3-70b-versatile",
    temperature=0
)
print("Connected to Groq LLM.")

def run_spark_sql(query: str) -> str:
    """Runs a Spark SQL query against the 'sales_records' view and returns results as text."""
    try:
        clean_query = query.strip().strip("```sql").strip("```").strip()
        result_df = spark.sql(clean_query)
        rows = result_df.limit(20).collect()
        if not rows:
            return "Query ran successfully but returned no rows."
        return "\n".join([str(row.asDict()) for row in rows])
    except Exception as e:
        return f"SQL Error: {e}. Available tables: 'sales_records' (transaction_id, user_id, product_name, category, quantity, price_per_unit, total_amount, location_city, timestamp) and 'vendor_performance' (vendor_id, vendor_name, city, total_orders, defect_orders, avg_tracking_upload_hours, defect_rate_pct)."

spark_sql_tool = Tool(
    name="SparkSQLQuery",
    func=run_spark_sql,
    description=(
        "Use this tool to answer questions about sales, transactions, revenue, quantities, "
        "products, categories, cities, vendors, defect rates, or tracking delays. Input must "
        "be a valid Spark SQL query string (no markdown, no explanation, just the SQL). "
        "Two tables are available:\n"
        "1) 'sales_records': transaction_id, user_id, product_name, category, quantity, "
        "price_per_unit, total_amount, location_city, timestamp.\n"
        "2) 'vendor_performance': vendor_id, vendor_name, city, total_orders, defect_orders, "
        "avg_tracking_upload_hours, defect_rate_pct. Use this table for any question about "
        "vendor defect rates, ODR, or tracking upload delays."
    )
)

def run_policy_search(query: str) -> str:
    """Searches the Qdrant vector store for relevant policy/FAQ text."""
    docs = vector_store.similarity_search(query, k=2)
    if not docs:
        return "No relevant policy information found."
    return "\n\n".join([f"[Source: {d.metadata['source']}]\n{d.page_content}" for d in docs])

policy_search_tool = Tool(
    name="PolicySearch",
    func=run_policy_search,
    description=(
        "Use this tool to answer questions about company policies, shipping rules, "
        "vendor agreements, return/refund rules, or product FAQs. Input should be a "
        "natural language question."
    )
)

tools = [spark_sql_tool, policy_search_tool]

react_prompt = PromptTemplate.from_template("""
You are OmniCart's internal analytics and policy assistant. Answer the user's
question using the tools available. Always pick the tool that matches the
question type: SparkSQLQuery for numeric/transactional questions, PolicySearch
for policy/FAQ questions.

TOOLS:
------
{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
""")

agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

print("Agent ready.")

print("\n" + "="*70)
print("TEST 1: Structured query")
print("="*70)
response1 = agent_executor.invoke({
    "input": "What is the total revenue generated in Lahore?"
})
print("\nFINAL ANSWER:", response1["output"])

print("\n" + "="*70)
print("TEST 2: Policy/RAG query")
print("="*70)
response2 = agent_executor.invoke({
    "input": "What happens if a vendor is late uploading tracking information?"
})
print("\nFINAL ANSWER:", response2["output"])

Connected to Groq LLM.
Agent ready.

TEST 1: Structured query


> Entering new AgentExecutor chain...
Thought: To find the total revenue generated in Lahore, I need to analyze the sales data for the city of Lahore. This requires a query on the sales records to sum up the total amount of all transactions made in Lahore.

Action: SparkSQLQuery
Action Input: SELECT SUM(total_amount) FROM sales_records WHERE location_city = 'Lahore'{'sum(total_amount)': 1601841.6500000001}Thought: I now know the final answer
Final Answer: The total revenue generated in Lahore is 1601841.65.

> Finished chain.

FINAL ANSWER: The total revenue generated in Lahore is 1601841.65.

TEST 2: Policy/RAG query


> Entering new AgentExecutor chain...
Thought: The question is about the policy regarding vendors who are late uploading tracking information, so I should use the PolicySearch tool to find the answer.

Action: PolicySearch
Action Input: What happens if a vendor is late uploading tracking information?[Source

In [ ]:
import gradio as gr

agent_executor.return_intermediate_steps = True

ENGINE_LABELS = {
    "SparkSQLQuery": "Structured Engine (Spark SQL)",
    "PolicySearch": "Unstructured Engine (Qdrant RAG)"
}

def ask_omnicart(message):
    try:
        response = agent_executor.invoke({"input": message})
        answer = response["output"]

        steps = response.get("intermediate_steps", [])
        tools_used = []
        for action, _observation in steps:
            tool_name = action.tool
            label = ENGINE_LABELS.get(tool_name, tool_name)
            if label not in tools_used:
                tools_used.append(label)

        engine_tag = " + ".join(tools_used) if tools_used else "Direct Answer"
        return answer, f"Currently using: {engine_tag}"

    except Exception as e:
        return f"Something went wrong: {e}", "Currently using: N/A"


example_questions = [
    "What is the total revenue generated in Lahore?",
    "Which product category has the highest sales in Karachi?",
    "What happens if a vendor is late uploading tracking information?",
    "How long does standard shipping take?",
    "Why might Fast Fashion Lahore be at risk of suspension?",
    "What triggers a factory reset on the wireless buds?"
]

custom_css = """
.gradio-container, body, .app, .main {
    background: linear-gradient(160deg, #8f0f22 0%, #c0152f 100%) !important;
}
#component-0, .block, .form, .panel {
    background: transparent !important;
}

/* Chat outer container */
.chatbot, div[data-testid="chatbot"],
[class*="chatbot"], [class*="bubble-wrap"] {
    background: transparent !important;
    border: 1px solid rgba(255,255,255,0.3) !important;
}

/* Chatbot header label pill (top-left "OmniCart Assistant" tag) */
.label-wrap, [class*="label-wrap"],
.chatbot .label, [data-testid="chatbot"] .label,
.chatbot > div:first-child,
.block-label, [class*="block-label"] {
    background-color: transparent !important;
    color: #ffffff !important;
    border: 1px solid rgba(255,255,255,0.4) !important;
}
.label-wrap svg, [class*="label-wrap"] svg,
.block-label svg, [class*="block-label"] svg {
    fill: #ffffff !important;
}

/* Clean, flat, direction-aware chat bubbles */
.message-wrap .message,
.chatbot .message {
    border: none !important;
    border-radius: 16px !important;
    padding: 12px 18px !important;
    margin: 6px 0 !important;
    color: #ffffff !important;
    box-shadow: 0 2px 6px rgba(0,0,0,0.25) !important;
}

/* User bubble — lighter red, flattened bottom-right corner */
.message.user, .user .message {
    background-color: #e5484d !important;
    border-bottom-right-radius: 4px !important;
}

/* Assistant bubble — deep red, flattened bottom-left corner */
.message.bot, .bot .message {
    background-color: #7a0d1e !important;
    border-bottom-left-radius: 4px !important;
}

/* Remove avatars entirely */
.avatar-container, .avatar, [class*="avatar"],
img[class*="avatar"] {
    display: none !important;
}

/* Text input box */
textarea, input[type="text"], input[type="email"] {
    background-color: rgba(255,255,255,0.1) !important;
    color: #ffffff !important;
    border: 1px solid rgba(255,255,255,0.5) !important;
    border-radius: 8px !important;
}
textarea::placeholder, input::placeholder {
    color: rgba(255,255,255,0.7) !important;
}

/* Icon buttons (copy, like/dislike, fullscreen, share, delete) */
button.icon, .icon-button, [class*="icon-button"],
button[aria-label] {
    background-color: transparent !important;
    box-shadow: none !important;
}
button.icon svg, .icon-button svg, [class*="icon-button"] svg,
button[aria-label] svg {
    fill: #ffffff !important;
    color: #ffffff !important;
}
/* Remove any blockquote / list decorative borders inherited from markdown styling */
.message blockquote, .message ul, .message ol, .message li,
.prose blockquote, .prose ul, .prose ol, .prose li {
    border-left: none !important;
    border: none !important;
    padding-left: 0 !important;
    margin-left: 0 !important;
}

/* Buttons */
button.primary {
    background-color: #ffffff !important;
    color: #7a0d1e !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
}
button.secondary {
    background-color: transparent !important;
    color: #ffffff !important;
    border: 1px solid #ffffff !important;
    border-radius: 8px !important;
}

/* All text white */
h1, h2, h3, h4, h5, h6,
p, span, div, label,
.prose, .prose p, .prose li,
label > span, strong, b,
.prose strong, .prose b {
    color: #ffffff !important;
}

/* Status line under title */
#engine-status {
    font-size: 0.95em !important;
    opacity: 0.9 !important;
    font-style: italic !important;
}
"""

with gr.Blocks(title="OmniCart Analytics AI", css=custom_css) as demo:
    gr.Markdown("# OmniCart Analytics AI")
    gr.Markdown(
        "**Enterprise Big Data + RAG Assistant** — ask about sales data or company policies, "
        "and the system automatically routes your question to the right engine."
    )
    engine_status = gr.Markdown("Currently using: —", elem_id="engine-status")

    chatbot = gr.Chatbot(height=450, label="OmniCart Assistant", avatar_images=None)
    msg = gr.Textbox(
        label="Ask a question",
        placeholder="e.g. What is the total revenue in Karachi?",
        lines=1
    )

    with gr.Row():
        submit_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear Chat", variant="secondary")

    gr.Markdown("**Try an example:**")
    with gr.Row():
        example_buttons = [gr.Button(q, size="sm", variant="secondary") for q in example_questions]

    def user_submit(user_message, chat_history):
        chat_history = chat_history + [{"role": "user", "content": user_message}]
        return "", chat_history

    def bot_respond(chat_history):
        user_message = chat_history[-1]["content"]
        bot_message, status = ask_omnicart(user_message)
        chat_history = chat_history + [{"role": "assistant", "content": bot_message}]
        return chat_history, status

    msg.submit(user_submit, [msg, chatbot], [msg, chatbot]).then(
        bot_respond, chatbot, [chatbot, engine_status]
    )
    submit_btn.click(user_submit, [msg, chatbot], [msg, chatbot]).then(
        bot_respond, chatbot, [chatbot, engine_status]
    )
    clear_btn.click(lambda: ([], "Currently using: —"), None, [chatbot, engine_status])

    for btn, q in zip(example_buttons, example_questions):
        btn.click(
            lambda question=q: question, None, msg
        ).then(
            user_submit, [msg, chatbot], [msg, chatbot]
        ).then(
            bot_respond, chatbot, [chatbot, engine_status]
        )

print("Launching Gradio dashboard...")
demo.launch(share=True, debug=True)